# Boat Party Ticket: Round 1 research and Round 2 strategy design

This notebook is research-only. It does not import or modify `trader_interface/algorithm.py`. The backtest uses the competition clock exactly: a position chosen after observing price `t` earns `position[t] * (price[t+1] - price[t])`; day 364 has no following-day P&L.

The centered Round 1 smooths and any template derived from it are retrospective, in-sample objects. They are tested as a possible fixed prior for Round 2, not presented as a causal Round 1 signal. Causal claims are reserved for calendar rules and online updates.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'trader_interface/data/Boat Party Ticket_price_history.csv').exists()), None)
assert root is not None, 'repository root not found'
sys.path.insert(0, str(root / 'research/boat_party'))
import analysis as a

out_dir = root / 'research/boat_party'
figure_dir = out_dir / 'figures'
result_dir = out_dir / 'results'
figure_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)
prices = a.load_prices(root).Price.to_numpy(dtype=float)
assert len(prices) == 365
print(f'root={root}')
print(f'prices={len(prices)}, min={prices.min():.2f}, max={prices.max():.2f}, mean={prices.mean():.2f}')

root=D:\Documents\Algojam
prices=365, min=39.58, max=55.39, mean=46.85


## Official calendar inputs

The dates below are transcribed from the official UQ academic calendar: [UQ Academic calendar](https://about.uq.edu.au/academic-calendar). Day zero is provisionally treated as 1 January for alignment diagnostics only.

In [2]:
calendar = a.calendar_events_frame()
display(calendar[['year', 'semester', 'event', 'dates', 'start_day', 'end_day']])
comparison = a.calendar_comparison()
display(comparison)
calendar.to_csv(result_dir / 'calendar_events.csv', index=False)

,year,semester,event,dates,start_day,end_day
0,2026,S1,Orientation Week,16-20 Feb 2026,46,50
1,2026,S1,Classes commence,23 Feb 2026,53,53
2,2026,S1,In-semester examinations,27-29 Mar; 17-19 Apr; 2 May 2026,85,121
3,2026,S1,In-semester break,6-12 Apr 2026,95,101
4,2026,S1,Classes resume,13 Apr 2026,102,102
5,2026,S1,Revision period,1-5 Jun 2026,151,155
6,2026,S1,Final examination period,6-20 Jun 2026,156,170
7,2026,S1,Semester ending,20 Jun 2026,170,170
8,2026,S2,Orientation Week,20-24 Jul 2026,200,204
9,2026,S2,Classes commence,27 Jul 2026,207,207


,landmark,day_2026,day_2027,2027_minus_2026
0,s1_orientation_mid,48.0,47.0,-1.0
1,s1_classes,53.0,52.0,-1.0
2,s1_break,95.0,84.0,-11.0
3,s1_resume,102.0,94.0,-8.0
4,s1_exam_mid,163.0,162.0,-1.0
5,s1_end,170.0,169.0,-1.0
6,s2_orientation_mid,202.0,201.0,-1.0
7,s2_classes,207.0,206.0,-1.0
8,s2_break,270.0,269.0,-1.0
9,s2_resume,278.0,277.0,-1.0


## Descriptive analysis

The four broad waves are treated as approximate episodes, not exact fitted events. Volatility, level-to-next-move correlation, and residual diagnostics are reported by broad phase.

In [3]:
changes = np.diff(prices)
descriptive = pd.DataFrame({
    'n_prices': [len(prices)],
    'n_changes': [len(changes)],
    'price_mean': [prices.mean()],
    'price_std': [prices.std(ddof=1)],
    'change_mean': [changes.mean()],
    'change_std': [changes.std(ddof=1)],
    'change_lag1_acf': [np.corrcoef(changes[:-1], changes[1:])[0, 1]],
    'price_level_next_move_corr': [np.corrcoef(prices[:-1], changes)[0, 1]],
})
display(descriptive.round(4))
turning = a.local_turning_points(prices, window=14)
display(turning)
phase_volatility = a.volatility_by_phase(prices)
display(phase_volatility.round(3))
level_correlations = a.level_next_move_correlations(prices)
display(level_correlations.head(8).round(3))
template_14 = a.template_from_prices(prices, 14)
diagnostics = a.residual_diagnostics(prices, template_14)
display(diagnostics.round(4))

,n_prices,n_changes,price_mean,price_std,change_mean,change_std,change_lag1_acf,price_level_next_move_corr
0,365,364,46.8492,3.9271,-0.0015,1.0114,-0.1677,-0.1278


,day,kind,price,smoothed_price,date_if_2026_day0
0,1,trough,44.69,44.945333,2026-01-02
1,47,peak,54.34,54.516000,2026-02-17
2,94,trough,45.45,44.869333,2026-04-05
3,112,peak,50.06,48.409333,2026-04-23
4,140,trough,43.06,43.012000,2026-05-21
5,161,trough,40.26,40.593333,2026-06-11
6,195,peak,54.82,54.010000,2026-07-15
7,234,trough,45.97,46.184667,2026-08-23
8,252,peak,49.93,49.864000,2026-09-10
9,279,trough,43.29,42.886000,2026-10-07


,phase,days,std_change,mean_abs_change,mean_change
0,summer_pre_s1,46,0.908,0.599,0.213
1,s1_orientation,5,0.599,0.464,-0.004
2,s1_classes_pre_break,44,0.803,0.631,-0.211
3,s1_break,7,0.629,0.490,-0.079
4,s1_classes_post_break,46,1.040,0.751,0.017
5,s1_revision_exam,23,1.078,0.591,-0.179
6,winter_break,29,1.490,0.874,0.422
7,s2_orientation,5,1.233,0.668,-0.320
8,s2_classes_pre_break,65,0.945,0.612,-0.124
9,s2_break,8,0.865,0.695,-0.100


,regime,n,corr
0,all,364,-0.128
1,low_quartile,92,0.103
2,middle_half,182,-0.032
3,high_quartile,90,-0.052
4,summer_pre_s1,46,-0.138
5,s1_orientation,5,-0.930
6,s1_classes_pre_break,44,-0.234
7,s1_break,7,-0.852


,series,n,mean,std,phi_ar1,half_life_days,lag1_acf
0,raw_level_minus_45_all,365,1.8492,3.9271,0.9671,20.7222,0.9669
1,template_residual_all,365,0.0013,0.9042,0.4317,0.8252,0.4317
2,raw_level_minus_45_d302,63,44.3254,1.6258,0.8219,3.5346,0.8549
3,summer_level_minus_45,63,-0.6746,1.6258,0.8219,3.5346,0.8549
4,summer_level_next_move_d302,62,44.3254,1.6258,0.8219,3.5346,-0.3362
5,summer_level_next_move_d322,42,45.0421,0.3949,-0.0858,0.2823,-0.7273


## Exact-timing backtest and broad calendar benchmark

The benchmark is deliberately broad: flat in the early and late summer, long into each wave, and short after each broad peak. Boundary sweeps are shown as sensitivity evidence, not as permission to choose the single best Round 1 shift.

In [4]:
schedule_rows = []
for shift in range(-14, 15):
    for buffer in [0, 2, 3, 5, 7]:
        for reduced in [False, True]:
            pos = a.fixed_schedule_positions(shift=shift, boundary_buffer=buffer, reduced_boundary=reduced)
            r = a.backtest(prices, pos, label='fixed_schedule')
            schedule_rows.append({'shift': shift, 'buffer': buffer, 'reduced_boundary': reduced, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown'], 'active_days': r['active_days']})
schedule_frame = pd.DataFrame(schedule_rows)
display(schedule_frame.sort_values('pnl', ascending=False).head(12))
display(schedule_frame.groupby(['buffer', 'reduced_boundary'])['pnl'].agg(['mean', 'median', 'min', 'max']).round(0))

local_rows = []
for local_shift in range(-14, 15):
    pos = a.fixed_schedule_positions(local_easter_shift=local_shift)
    r = a.backtest(prices, pos, label='local_easter')
    local_rows.append({'local_easter_shift': local_shift, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown']})
local_frame = pd.DataFrame(local_rows)
display(local_frame.iloc[::4].round(0))

base_schedule = a.fixed_schedule_positions()
calendar_schedule = a.calendar_schedule_positions(2027)
schedule_results = [a.backtest(prices, base_schedule, 'broad_fixed_schedule'), a.backtest(prices, calendar_schedule, 'calendar_2027_schedule')]
display(a.metrics_frame(schedule_results).round(3))

,shift,buffer,reduced_boundary,pnl,max_drawdown,active_days
111,-3,0,True,71640.0,-3460.0,287
110,-3,0,False,71640.0,-3460.0,287
130,-1,0,False,68370.0,-3460.0,287
131,-1,0,True,68370.0,-3460.0,287
121,-2,0,True,66690.0,-3460.0,287
120,-2,0,False,66690.0,-3460.0,287
133,-1,2,True,66690.0,-3460.0,287
113,-3,2,True,66575.0,-3460.0,287
100,-4,0,False,66420.0,-3460.0,287
101,-4,0,True,66420.0,-3460.0,287


mean   median     min      max
buffer reduced_boundary                                   
0      False             42843.0  42930.0 -2900.0  71640.0
       True              42843.0  42930.0 -2900.0  71640.0
2      False             41083.0  43630.0  6620.0  65010.0
       True              41963.0  44495.0  1860.0  66690.0
3      False             40127.0  41570.0  4140.0  62180.0
       True              41485.0  43200.0   620.0  65960.0
5      False             37644.0  41320.0  6680.0  56510.0
       True              40244.0  43485.0  1890.0  61600.0
7      False             33105.0  35800.0 -6190.0  47150.0
       True              37974.0  38475.0 -4545.0  56570.0

,local_easter_shift,pnl,max_drawdown
0,-14,41970.0,-9190.0
4,-10,49790.0,-7920.0
8,-6,60510.0,-3460.0
12,-2,64250.0,-3460.0
16,2,66130.0,-3460.0
20,6,63590.0,-3630.0
24,10,58630.0,-3630.0
28,14,43570.0,-6860.0


,model,pnl,sharpe,hit_rate,active_hit_rate,active_days,total_days,max_drawdown,max_capital,avg_active_capital,pnl_per_max_capital,budget_violations,integral_positions,within_limit,quarter_1_pnl,quarter_2_pnl,quarter_3_pnl,quarter_4_pnl
0,broad_fixed_schedule,65990.0,3.874,0.437,0.554,287,364,-3460.0,55390.0,47483.171,1.191,0,1,1,15920.0,24240.0,23730.0,2100.0
1,calendar_2027_schedule,60840.0,3.567,0.448,0.570,286,364,-3460.0,55390.0,47509.965,1.098,0,1,1,11120.0,22000.0,24610.0,3110.0


## Fixed and calendar-warped seasonal templates

A centered moving average gives a compact Round 1 seasonal prior. The calendar-warped version maps 2027 event landmarks back onto the 2026 template with a monotone piecewise-linear map. The fixed and warped versions are both evaluated on Round 1 only as retrospective diagnostics.

In [5]:
template_rows = []
templates = {}
for window in [3, 5, 7, 10, 14, 21, 30]:
    template = a.template_from_prices(prices, window)
    templates[window] = template
    for deadband in [0.0, 0.02, 0.05, 0.10, 0.20, 0.30]:
        r = a.backtest(prices, a.template_positions(template, deadband), f'template_w{window}')
        template_rows.append({'window': window, 'deadband': deadband, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown'], 'active_days': r['active_days']})
template_frame = pd.DataFrame(template_rows)
display(template_frame.sort_values('pnl', ascending=False).head(15))

fixed_7 = templates[7]
warped_7 = a.warped_template(prices, 7, 2027)
blend_rows = []
for weight in [0.0, 0.25, 0.50, 0.75, 1.0]:
    for disagreement in [True, False]:
        pos = a.blend_template_positions(fixed_7, warped_7, weight, disagreement)
        r = a.backtest(prices, pos, 'template_blend')
        blend_rows.append({'weight_warped': weight, 'flat_on_disagreement': disagreement, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown'], 'active_days': r['active_days']})
blend_frame = pd.DataFrame(blend_rows)
display(blend_frame.round(0))

,window,deadband,pnl,max_drawdown,active_days
13,7,0.02,88260.0,-5280.0,332
12,7,0.00,87510.0,-5280.0,364
4,3,0.20,85900.0,-3620.0,182
19,10,0.02,84420.0,-2720.0,327
18,10,0.00,83960.0,-3130.0,363
0,3,0.00,83940.0,-5760.0,362
14,7,0.05,83930.0,-5280.0,296
20,10,0.05,83860.0,-2720.0,291
15,7,0.10,83440.0,-4130.0,243
1,3,0.02,82950.0,-6460.0,338


,weight_warped,flat_on_disagreement,pnl,max_drawdown,active_days
0,0.0,True,80270.0,-2720.0,289
1,0.0,False,87510.0,-5280.0,364
2,0.0,True,80270.0,-2720.0,289
3,0.0,False,83690.0,-5990.0,364
4,0.0,True,80270.0,-2720.0,289
5,0.0,False,85670.0,-5510.0,364
6,1.0,True,80270.0,-2720.0,289
7,1.0,False,73850.0,-6090.0,364
8,1.0,True,80270.0,-2720.0,289
9,1.0,False,73030.0,-3410.0,364


## Online amplitude/baseline, residual, Fourier, and calendar-regression tests

RLS and the compact Kalman filter below update only with observations through the decision day. Their Round 1 P&L is still not an unseen-year estimate because the external template was built from all of Round 1. Fourier and phase-regression models are causal expanding fits but have only one annual cycle to identify their coefficients.

In [6]:
model_positions = {
    'broad_fixed_schedule': base_schedule,
    'calendar_2027_schedule': calendar_schedule,
    'fixed_template_w7_db02': a.template_positions(fixed_7, 0.02),
    'fixed_template_w10_db02': a.template_positions(templates[10], 0.02),
    'template_shared_sign_w7': a.blend_template_positions(fixed_7, warped_7, 0.5, True),
    'rls_global_w7': a.rls_positions(prices, fixed_7, 0.995, 'global'),
    'rls_dual_template_agreement': a.rls_dual_template_agreement_positions(prices, fixed_7, warped_7, 0.995),
    'rls_separate_amplitudes_w7': a.rls_positions(prices, fixed_7, 0.995, 'separate_wave_amplitudes'),
    'kalman_residual_w7': a.kalman_residual_positions(prices, fixed_7, 0.001, 1.0),
    'fourier_k2': a.fourier_positions(prices, 2, 60, 1.0, False),
    'fourier_k2_ar1': a.fourier_positions(prices, 2, 60, 1.0, True),
    'calendar_phase_regression': a.phase_regression_positions(prices, warmup=60),
    'summer_fixed_45_d302': a.summer_reversion_positions(prices, 302, 45.0, False, 0),
    'summer_fixed_45_d322': a.summer_reversion_positions(prices, 322, 45.0, False, 0),
}
# A structurally simple combined rule: calendar schedule, then explicit summer reversion only after d302.
summer = a.summer_reversion_positions(prices, 302, 45.0, False, 7)
model_positions['schedule_plus_summer_45'] = np.where(summer != 0, summer, base_schedule)
residual_phi = a.ar1_phi(prices - fixed_7)
model_positions['fixed_template_plus_residual'] = a.residual_reversion_positions(prices, fixed_7, residual_phi)
model_positions['summer_online_baseline_d302'] = a.summer_reversion_positions(prices, 302, 45.0, True, 0)
model_positions['summer_smooth_d302'] = a.summer_reversion_positions(prices, 302, 45.0, False, 28)
model_positions['summer_smooth_d322'] = a.summer_reversion_positions(prices, 322, 45.0, False, 28)
model_results = {name: a.backtest(prices, pos, name) for name, pos in model_positions.items()}
model_frame = a.metrics_frame(model_results.values())
display(model_frame.sort_values('pnl', ascending=False).round(3))

residual_phi = a.ar1_phi(prices - fixed_7)
print(f'template residual phi={residual_phi:.3f}, half-life={a.half_life(residual_phi):.2f} days')
for order in [1, 2, 3, 4]:
    r = a.backtest(prices, a.fourier_positions(prices, order, 60, 1.0, False), f'fourier_{order}')
    print(f'Fourier order {order}: P&L={r["pnl"]:.0f}, drawdown={r["max_drawdown"]:.0f}')

,model,pnl,sharpe,hit_rate,active_hit_rate,active_days,total_days,max_drawdown,max_capital,avg_active_capital,pnl_per_max_capital,budget_violations,integral_positions,within_limit,quarter_1_pnl,quarter_2_pnl,quarter_3_pnl,quarter_4_pnl
15,fixed_template_plus_residual,164290.00,9.531,0.731,0.731,364,364,-1570.00,55390.0,46855.797,2.966,0,1,1,32880.0,47420.0,42570.0,41420.00
5,rls_global_w7,163850.00,9.499,0.728,0.728,364,364,-1560.00,55390.0,46855.797,2.958,0,1,1,30760.0,48620.0,42530.0,41940.00
8,kalman_residual_w7,163050.00,9.441,0.723,0.723,364,364,-1570.00,55390.0,46855.797,2.944,0,1,1,32880.0,46940.0,42870.0,40360.00
6,rls_dual_template_agreement,151000.00,8.908,0.604,0.743,296,364,-1560.00,55390.0,47035.000,2.726,0,1,1,28330.0,42320.0,40780.0,39570.00
7,rls_separate_amplitudes_w7,145360.00,8.214,0.668,0.669,363,364,-5940.00,55390.0,46860.909,2.624,0,1,1,23910.0,46080.0,43770.0,31600.00
2,fixed_template_w7_db02,88260.00,4.803,0.514,0.563,332,364,-5280.00,55390.0,46967.892,1.593,0,1,1,15290.0,28720.0,21420.0,22830.00
14,schedule_plus_summer_45,85868.31,4.771,0.555,0.579,349,364,-3460.00,55390.0,46576.143,1.550,0,1,1,15920.0,24240.0,23730.0,21978.31
3,fixed_template_w10_db02,84420.00,4.583,0.505,0.563,327,364,-2720.00,55390.0,46942.018,1.524,0,1,1,20270.0,31410.0,20190.0,12550.00
4,template_shared_sign_w7,80270.00,4.455,0.440,0.554,289,364,-2720.00,55390.0,46734.187,1.449,0,1,1,13630.0,27060.0,22380.0,17200.00
0,broad_fixed_schedule,65990.00,3.874,0.437,0.554,287,364,-3460.00,55390.0,47483.171,1.191,0,1,1,15920.0,24240.0,23730.0,2100.00


template residual phi=0.112, half-life=0.32 days
Fourier order 1: P&L=-250, drawdown=-15040
Fourier order 2: P&L=11350, drawdown=-14980
Fourier order 3: P&L=4030, drawdown=-23980
Fourier order 4: P&L=470, drawdown=-21280


## Semester transfer and leave-one-wave-out checks

The shape comparisons remove vertical amplitude before comparing the large and small waves. Leave-one-wave-out uses the median normalised shape of the other three episodes to determine the held-out directional sign; it is a diagnostic, not a production algorithm.

In [7]:
transfer_frame = a.semester_transfer(prices)
wave_frame = a.leave_one_wave_out(prices)
display(transfer_frame.round(3))
display(wave_frame.round(3))

,comparison,corr_normalised_shape,rmse_normalised_shape,amplitude_S1,amplitude_S2,amplitude_ratio_S2_over_S1
0,S1_vs_S2_large,0.469,0.446,9.874,14.093,1.427
1,S1_vs_S2_small,0.889,0.140,9.240,9.993,1.081
2,S1_large_vs_small,0.839,0.302,9.874,9.240,0.936
3,S2_large_vs_small,-0.111,0.727,14.093,9.993,0.709


,held_out_wave,fit_waves,pnl,active_hit_rate,max_drawdown,active_days
0,S1_large,"S1_small,S2_large,S2_small",13880.0,0.545,-1880.0,77
1,S1_small,"S1_large,S2_large,S2_small",14800.0,0.536,-3930.0,69
2,S2_large,"S1_large,S1_small,S2_small",14800.0,0.514,-3220.0,74
3,S2_small,"S1_large,S1_small,S2_large",3670.0,0.567,-5590.0,67


## Fixed-seed stress scenarios

Stress paths scale the large and small seasonal components from 60% to 140%, shift the summer baseline by AUD 1-2, vary the summer transition speed, and replace residuals with fixed-seed length-7 block bootstrap draws. The block bootstrap preserves short-run dependence better than an IID bootstrap. The stress ranking is more relevant to Round 2 than a single Round 1 maximum.

In [8]:
stress_rules = {
    'broad_fixed_schedule': lambda path: a.fixed_schedule_positions(),
    'calendar_2027_schedule': lambda path: a.calendar_schedule_positions(2027),
    'fixed_template_w7_db02': lambda path: a.template_positions(fixed_7, 0.02),
    'template_shared_sign_w7': lambda path: a.blend_template_positions(fixed_7, warped_7, 0.5, True),
    'rls_global_w7': lambda path: a.rls_positions(path, fixed_7, 0.995, 'global'),
    'rls_dual_template_agreement': lambda path: a.rls_dual_template_agreement_positions(path, fixed_7, warped_7, 0.995),
    'schedule_plus_summer_45': lambda path: np.where(a.summer_reversion_positions(path, 302, 45.0, False, 7) != 0, a.summer_reversion_positions(path, 302, 45.0, False, 7), a.fixed_schedule_positions()),
}
stress_summary, stress_detail = a.run_stress(prices, fixed_7, stress_rules, n_bootstrap=80, seed=a.SEED)
display(stress_summary.round(2))
display(stress_detail[stress_detail.scenario.str.contains('common_amp|summer_shift')].pivot(index='scenario', columns='model', values='pnl').round(0).head(12))

,model,n_scenarios,median_pnl,p10_pnl,worst_pnl,positive_rate,median_max_drawdown
4,rls_global_w7,96,169024.14,144463.57,130634.29,1.0,-5482.14
3,rls_dual_template_agreement,96,157899.29,136901.86,126977.14,1.0,-5152.86
2,fixed_template_w7_db02,96,89247.14,73213.57,40771.43,1.0,-7832.86
5,schedule_plus_summer_45,96,85645.27,73112.60,61576.31,1.0,-6352.28
6,template_shared_sign_w7,96,79836.43,64480.86,33004.29,1.0,-6449.29
0,broad_fixed_schedule,96,61101.00,52985.71,41698.00,1.0,-6232.14
1,calendar_2027_schedule,96,59846.43,52062.86,37131.43,1.0,-5922.14


model,broad_fixed_schedule,calendar_2027_schedule,fixed_template_w7_db02,rls_dual_template_agreement,rls_global_w7,schedule_plus_summer_45,template_shared_sign_w7
scenario,,,,,,,
common_amp_0.6,41698.0,37131.0,52744.0,135626.0,144139.0,61576.0,48692.0
common_amp_0.8,53844.0,48986.0,70502.0,145865.0,153763.0,73722.0,64481.0
common_amp_1.0,65990.0,60840.0,88260.0,151000.0,163850.0,85868.0,80270.0
common_amp_1.2,78136.0,72694.0,106018.0,155021.0,166235.0,98014.0,96059.0
common_amp_1.4,90282.0,84549.0,123776.0,160650.0,169291.0,110160.0,111848.0
summer_shift_+1.0,65919.0,60840.0,89117.0,135726.0,146413.0,72791.0,81270.0
summer_shift_+2.0,65847.0,60840.0,89974.0,135814.0,148379.0,73434.0,82270.0
summer_shift_-1.0,66061.0,60840.0,87403.0,134866.0,149296.0,71385.0,79270.0
summer_shift_-2.0,66133.0,60840.0,86546.0,129709.0,143181.0,68523.0,78270.0


## Reproducible outputs and validity audit

The figures and CSV tables are intentionally small. The final report distinguishes the in-sample template evidence from causal calendar evidence and scenario stress.

In [9]:
rls_rows = []
for mode in ['global', 'separate_wave_amplitudes']:
    for forgetting in [0.98, 0.995, 1.0]:
        r = a.backtest(prices, a.rls_positions(prices, fixed_7, forgetting, mode), f'rls_{mode}')
        rls_rows.append({'mode': mode, 'forgetting': forgetting, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown'], 'active_days': r['active_days']})
rls_sensitivity = pd.DataFrame(rls_rows)
display(rls_sensitivity.round(2))
rls_sensitivity.to_csv(result_dir / 'rls_sensitivity.csv', index=False)

def shifted_template(template, shift):
    grid = np.arange(len(template), dtype=float)
    return np.interp(grid + shift, grid, template, left=template[0], right=template[-1])
perturb_rows = []
for delta in [-14, -7, -3, -1, 0, 1, 3, 7, 14]:
    fixed_shifted = shifted_template(fixed_7, delta)
    warp_shifted = shifted_template(warped_7, delta)
    for name, pos in {
        'fixed_template': a.template_positions(fixed_shifted, 0.02),
        'shared_sign_blend': a.blend_template_positions(fixed_shifted, warp_shifted, 0.5, True),
    }.items():
        r = a.backtest(prices, pos, name)
        perturb_rows.append({'calendar_shift': delta, 'model': name, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown']})
perturb_frame = pd.DataFrame(perturb_rows)
display(perturb_frame.pivot(index='calendar_shift', columns='model', values='pnl').round(0))
perturb_frame.to_csv(result_dir / 'calendar_perturbation.csv', index=False)

adapt_rows = []
for amp in [0.6, 1.0, 1.4]:
    path = a.scenario_price(prices, fixed_7, amp, amp)
    for name, pos in {
        'fixed_template': a.template_positions(fixed_7, 0.02),
        'rls_global': a.rls_positions(path, fixed_7, 0.995, 'global'),
    }.items():
        daily = pos[:-1].astype(float) * np.diff(path)
        for horizon in [30, 60, 90, 365]:
            adapt_rows.append({'amplitude': amp, 'model': name, 'horizon_days': horizon, 'pnl': float(daily[:horizon].sum())})
adaptation_frame = pd.DataFrame(adapt_rows)
adaptation_summary = adaptation_frame.groupby(['model', 'horizon_days'])['pnl'].agg(['median', 'min', 'max']).reset_index()
display(adaptation_summary.round(0))
adaptation_frame.to_csv(result_dir / 'adaptation_reliability.csv', index=False)

model_frame = a.metrics_frame(model_results.values())
pseudo_2027 = np.interp(a.calendar_warp_indices(2026, 2027), np.arange(365), prices)
generator_rows = []
for generator, path in [('observed_fixed_day', prices), ('event_warped_pseudo_2027', pseudo_2027)]:
    generator_rules = {
        'fixed_schedule': a.fixed_schedule_positions(),
        'calendar_2027_schedule': a.calendar_schedule_positions(2027),
        'fixed_template': a.template_positions(fixed_7, 0.02),
        'warped_template': a.template_positions(warped_7, 0.02),
        'rls_fixed': a.rls_positions(path, fixed_7, 0.995, 'global'),
        'rls_warped': a.rls_positions(path, warped_7, 0.995, 'global'),
        'rls_dual_agreement': a.rls_dual_template_agreement_positions(path, fixed_7, warped_7, 0.995),
    }
    for name, pos in generator_rules.items():
        r = a.backtest(path, pos, name)
        generator_rows.append({'generator': generator, 'model': name, 'pnl': r['pnl'], 'max_drawdown': r['max_drawdown'], 'active_days': r['active_days']})
generator_frame = pd.DataFrame(generator_rows)
display(generator_frame.round(0))
generator_frame.to_csv(result_dir / 'generator_designs.csv', index=False)
shortlisted = {name: model_results[name] for name in ['broad_fixed_schedule', 'calendar_2027_schedule', 'fixed_template_w7_db02', 'template_shared_sign_w7', 'rls_global_w7', 'rls_dual_template_agreement']}
a.save_figures(prices, fixed_7, shortlisted, figure_dir)
a.write_result_tables(result_dir, model_frame, schedule_frame, template_frame, stress_summary, stress_detail, wave_frame, transfer_frame, diagnostics, calendar)

for name, pos in model_positions.items():
    audit = a.backtest(prices, pos, name)
    assert audit['integral_positions'] == 1
    assert audit['within_limit'] == 1
    assert audit['budget_violations'] == 0
print('validity audit passed for all model candidates')
print('figures:', sorted(str(p.relative_to(out_dir)) for p in figure_dir.glob('*.png')))
print('tables:', sorted(str(p.relative_to(out_dir)) for p in result_dir.glob('*.csv')))

,mode,forgetting,pnl,max_drawdown,active_days
0,global,0.98,161070.0,-1570.0,364
1,global,1.00,163850.0,-1560.0,364
2,global,1.00,163850.0,-1560.0,364
3,separate_wave_amplitudes,0.98,130560.0,-5940.0,363
4,separate_wave_amplitudes,1.00,145360.0,-5940.0,363
5,separate_wave_amplitudes,1.00,147040.0,-5940.0,363


model,fixed_template,shared_sign_blend
calendar_shift,,
-14,-11690.0,-2910.0
-7,10780.0,31970.0
-3,120050.0,102920.0
-1,76680.0,79260.0
0,88260.0,80270.0
1,76990.0,66820.0
3,119630.0,75100.0
7,36740.0,31300.0
14,1960.0,-6900.0


,model,horizon_days,median,min,max
0,fixed_template,30,1170.0,-1822.0,4162.0
1,fixed_template,60,8200.0,2904.0,13496.0
2,fixed_template,90,15750.0,7481.0,24019.0
3,fixed_template,365,88260.0,52744.0,123776.0
4,rls_global,30,9006.0,7897.0,9340.0
5,rls_global,60,19200.0,16237.0,20111.0
6,rls_global,90,30300.0,27470.0,32649.0
7,rls_global,365,163850.0,144139.0,169291.0


,generator,model,pnl,max_drawdown,active_days
0,observed_fixed_day,fixed_schedule,65990.0,-3460.0,287
1,observed_fixed_day,calendar_2027_schedule,60840.0,-3460.0,286
2,observed_fixed_day,fixed_template,88260.0,-5280.0,332
3,observed_fixed_day,warped_template,67460.0,-4120.0,325
4,observed_fixed_day,rls_fixed,163850.0,-1560.0,364
5,observed_fixed_day,rls_warped,138150.0,-1570.0,364
6,observed_fixed_day,rls_dual_agreement,151000.0,-1560.0,296
7,event_warped_pseudo_2027,fixed_schedule,60746.0,-3427.0,287
8,event_warped_pseudo_2027,calendar_2027_schedule,65439.0,-3330.0,286
9,event_warped_pseudo_2027,fixed_template,73247.0,-3539.0,332


validity audit passed for all model candidates
figures: ['figures\\disjoint_validation_cumulative_pnl.png', 'figures\\ewma_deviation_next_return.png', 'figures\\ewma_overlay_cumulative_pnl.png', 'figures\\ewma_parameter_sensitivity.png', 'figures\\ewma_segment_pnl.png', 'figures\\fixed_template_final_strategy.png', 'figures\\independent_stress_pnl.png', 'figures\\model_cumulative_pnl.png', 'figures\\price_calendar.png']
tables: ['results\\adaptation_reliability.csv', 'results\\agreement_policy_comparison.csv', 'results\\calendar_events.csv', 'results\\calendar_perturbation.csv', 'results\\ewma_alpha_threshold_mechanics.csv', 'results\\ewma_bootstrap_placebo.csv', 'results\\ewma_bootstrap_placebo_summary.csv', 'results\\ewma_chronological_splits.csv', 'results\\ewma_concentration.csv', 'results\\ewma_correctness_checks.csv', 'results\\ewma_denominator_comparison.csv', 'results\\ewma_overfitting_diagnostics.csv', 'results\\ewma_parameter_sensitivity.csv', 'results\\ewma_pnl_attribution.c

## Second-stage disjoint validation audit

The preceding sections are first-stage evidence. Any full-year centered-template score is **in-sample template reconstruction**, because the template was built from the scored Round 1 path. The original `stress_summary.csv` is **template-conditioned sensitivity**, because its paths were reconstructed from that same template; neither is out-of-sample validation.

The following section freezes the preregistered seven-day/RLS candidate and tests it with segment-local S1-to-S2 templates, leakage assertions, ablations, placebos, independent event-kernel and cross-semester generators, and standalone summer policies.

In [10]:
from validation_pipeline import run_stage2_validation

stage2 = run_stage2_validation(root, n_paths=400, seed=a.SEED)
print('leakage checks passed:', bool((stage2['leakage']['passed'] == 1).all()))
print('independent paths:', len(stage2['independent_detail']) // stage2['independent_detail']['model'].nunique())
display(stage2['heldout_rls'].query("direction == 'S1_to_S2' and window in ['whole_semester', 'year_end_including_summer']")[['model', 'window', 'pnl', 'active_hit_rate', 'active_days', 'max_drawdown', 'template_source']].sort_values(['window', 'pnl'], ascending=[True, False]).round(2))
display(stage2['ablations'][['model', 'pnl', 'incremental_pnl_vs_previous_ablation', 'active_hit_rate', 'max_drawdown']].round(2))
display(stage2['independent_summary'][['model', 'n_paths', 'median_pnl', 'p10_pnl', 'worst_pnl', 'positive_path_rate', 'median_max_drawdown']].round(2))
display(stage2['timing_summary'].query("model in ['fixed_template_rls', 'warped_template_rls', 'dual_template_agreement', 'broad_calendar_schedule']")[['timing_mode', 'model', 'n_paths', 'median_pnl', 'p10_pnl', 'worst_pnl']].round(2))
display(stage2['summer_summary'][['model', 'n_paths', 'median_pnl', 'p10_pnl', 'worst_pnl', 'positive_path_rate']].round(2))
print('new validation tables written under research/boat_party/results')

leakage checks passed: True
independent paths: 400


,model,window,pnl,active_hit_rate,active_days,max_drawdown,template_source
36,constant_45_mean_reversion,whole_semester,46610.0,0.53,165,-3770.0,fixed AUD 45 equilibrium; no adaptation
38,broad_calendar_schedule,whole_semester,36650.0,0.56,141,-3330.0,official 2026 academic landmarks and broad win...
44,calendar_plus_summer,whole_semester,36650.0,0.56,141,-3330.0,official broad calendar schedule plus fixed AU...
39,fixed_transfer_no_rls,whole_semester,32270.0,0.57,141,-4020.0,S1 prices only; local 7-day transfer; no RLS
41,fixed_transfer_rls,whole_semester,28750.0,0.53,165,-5640.0,S1 prices only; fixed-index local 7-day transf...
37,online_baseline_mean,whole_semester,28630.0,0.51,165,-5810.0,online EWMA baseline only; no seasonal template
43,dual_transfer_agreement,whole_semester,12350.0,0.51,109,-6500.0,agreement of independent fixed-index and acade...
40,baseline_only_rls,whole_semester,2110.0,0.48,165,-13210.0,constant AUD 45 template; global RLS intercept...
42,academic_transfer_rls,whole_semester,-4050.0,0.48,165,-16870.0,S1 prices only; official event-phase local 7-d...
45,constant_45_mean_reversion,year_end_including_summer,59030.0,0.58,203,-3770.0,fixed AUD 45 equilibrium; no adaptation


,model,pnl,incremental_pnl_vs_previous_ablation,active_hit_rate,max_drawdown
0,constant_45_mean_reversion,59030.0,NaN,0.58,-3770.0
1,online_baseline_mean,37450.0,-21580.0,0.55,-5810.0
2,broad_calendar_schedule,36650.0,-800.0,0.56,-3330.0
3,fixed_transfer_no_rls,32270.0,-4380.0,0.57,-4020.0
4,baseline_only_rls,6330.0,-25940.0,0.51,-13210.0
5,fixed_transfer_rls,35230.0,28900.0,0.53,-5640.0
6,academic_transfer_rls,9550.0,-25680.0,0.55,-16870.0
7,dual_transfer_agreement,22390.0,12840.0,0.56,-6500.0
8,calendar_plus_summer,49070.0,26680.0,0.60,-3330.0


,model,n_paths,median_pnl,p10_pnl,worst_pnl,positive_path_rate,median_max_drawdown
0,dual_template_agreement,400,71767.57,22789.95,-9727.82,0.99,-7038.83
1,fixed_template_rls,400,71697.25,22865.56,-9358.27,0.99,-7183.79
2,warped_template_rls,400,71170.31,23599.38,-10097.37,0.99,-7348.90
3,constant45_mean_reversion,400,55652.00,19061.64,2138.93,1.00,-6347.52
4,online_baseline_mean,400,54577.33,13078.38,-8329.92,0.98,-6367.48
5,baseline_only_rls,400,53755.98,8903.41,-14881.82,0.96,-7199.40
6,calendar_plus_summer,400,24432.55,7947.06,-20899.69,0.96,-7177.72
7,broad_calendar_schedule,400,17030.42,1625.68,-20305.94,0.92,-7177.72
8,broad_fixed_schedule,400,15412.03,-1041.94,-19116.62,0.88,-7333.45
9,fixed_template_no_rls,400,9948.55,-11968.18,-38000.67,0.72,-12885.19


,timing_mode,model,n_paths,median_pnl,p10_pnl,worst_pnl
0,official_2027,warped_template_rls,100,90113.86,31869.77,10634.12
1,official_2027,fixed_template_rls,100,90069.58,23766.68,13094.17
2,official_2027,dual_template_agreement,100,89431.70,27654.71,12131.79
6,easter_early_11,warped_template_rls,100,78885.90,29585.99,9146.55
7,easter_early_11,fixed_template_rls,100,77484.99,30368.40,9627.81
8,easter_early_11,dual_template_agreement,100,77283.46,29763.13,9387.18
11,fixed_2026,fixed_template_rls,100,73404.36,23856.38,-3713.22
12,fixed_2026,dual_template_agreement,100,70930.60,22107.64,-4547.20
14,fixed_2026,warped_template_rls,100,69208.32,21629.40,-5381.17
15,easter_early_8,warped_template_rls,100,48280.54,15830.55,-10097.37


,model,n_paths,median_pnl,p10_pnl,worst_pnl,positive_path_rate
0,shrunk_mean,400,11262.77,3314.32,-1090.95,0.99
1,online_median,400,11013.28,1833.37,-4470.94,0.96
2,online_mean,400,7186.59,434.41,-1977.88,0.95
3,fixed,400,6056.47,296.71,-3577.04,0.94


new validation tables written under research/boat_party/results


## Final fixed-template candidate audit (frozen, research-only)

This section is the final validation pass requested for production preparation. It uses fixed day indices, the complete Round 1 path as an externally fixed Round 2 prior, centred 5-, 7- and 11-day templates, one-day template slopes, the frozen AUD 0.02 deadband, and the frozen day-322 AUD 45 summer switch. The prior report's labelled 10-day input was the helper's effective 11-day smoother; this section explicitly requests 11 and checks legacy position/P&L equivalence. It also evaluates paired 7-, 14- and 28-day transitions from AUD 45 to AUD 43, 45 and 47. It does not use calendar warping, RLS/Kalman/OU/Fourier models, extra parameter searches, or future values from evaluated paths. Same-year scores are labelled in-sample reconstruction; synthetic results are generator-conditioned stress diagnostics, not confidence intervals.

In [11]:
from final_strategy_test import run_final_strategy_test

final_test = run_final_strategy_test(root, n_noise_paths=400, n_summer_equilibrium_paths=100, n_gradual_summer_paths=200, seed=20260809)
display(final_test['comparison'][['model', 'evidence_label', 'pnl', 'sharpe', 'active_hit_rate', 'active_days', 'max_drawdown', 'max_capital', 'pnl_per_max_capital']].round(2))
display(final_test['timing_summary'][['model', 'zero_shift_pnl', 'worst_shift_pnl', 'median_shift_pnl', 'max_drawdown_across_shifts', 'max_percentage_loss_vs_zero']].round(2))
display(final_test['noise_summary'][['model', 'n_paths', 'median_pnl', 'p10_pnl', 'worst_pnl', 'positive_path_rate', 'median_max_drawdown']].round(2))
display(final_test['summer_summary'][['summer_equilibrium', 'model', 'n_paths', 'median_pnl', 'p10_pnl', 'worst_pnl', 'positive_path_rate', 'median_max_drawdown']].round(2))
display(final_test['gradual_summary'][['scenario_scope', 'target_equilibrium', 'transition_days', 'model', 'median_full_year_pnl', 'p10_full_year_pnl', 'worst_full_year_pnl', 'median_summer_only_pnl', 'p10_summer_only_pnl', 'worst_summer_only_pnl', 'median_max_drawdown', 'positive_path_rate', 'paired_median_d_minus_b', 'paired_p10_d_minus_b', 'paired_worst_d_minus_b']].round(2))
display(final_test['gradual_paired'][['scenario_scope', 'target_equilibrium', 'transition_days', 'n_paths', 'paired_median_d_minus_b', 'paired_p10_d_minus_b', 'paired_worst_d_minus_b']].round(2))
display(final_test['selection'])
print('Final fixed 5/7/11 audit complete; outputs written under research/boat_party/results')

Round 1 prices: 365
                            model     pnl   sharpe  active_hit_rate  active_days  max_drawdown  max_capital
                      Candidate A 88260.0 4.803340         0.563253          332       -5280.0      55390.0
                      Candidate B 84040.0 4.595244         0.559006          322       -3150.0      55390.0
                      Candidate C 94910.0 5.165312         0.585294          340       -5280.0      55390.0
                      Candidate D 92560.0 5.053612         0.582583          333       -2460.0      55390.0
         Broad fixed-day schedule 65990.0 3.874076         0.554007          287       -3460.0      55390.0
   Constant AUD 45 mean reversion 79060.0 4.201829         0.573003          363      -10260.0      55390.0
         Flat Boat Party position     0.0 0.000000              NaN            0           0.0          0.0
Fixed 7-day template, no deadband 87510.0 4.675892         0.563187          364       -5280.0      55390.0
correctn

,model,evidence_label,pnl,sharpe,active_hit_rate,active_days,max_drawdown,max_capital,pnl_per_max_capital
0,Candidate A,in-sample Round 1 reconstruction,88260.0,4.80,0.56,332,-5280.0,55390.0,1.59
1,Candidate B,in-sample Round 1 reconstruction,84040.0,4.60,0.56,322,-3150.0,55390.0,1.52
2,Candidate C,in-sample Round 1 reconstruction,94910.0,5.17,0.59,340,-5280.0,55390.0,1.71
3,Candidate D,in-sample Round 1 reconstruction,92560.0,5.05,0.58,333,-2460.0,55390.0,1.67
4,Broad fixed-day schedule,in-sample Round 1 reconstruction,65990.0,3.87,0.55,287,-3460.0,55390.0,1.19
5,Constant AUD 45 mean reversion,in-sample Round 1 reconstruction,79060.0,4.20,0.57,363,-10260.0,55390.0,1.43
6,Flat Boat Party position,in-sample Round 1 reconstruction,0.0,0.00,NaN,0,0.0,0.0,0.00
7,"Fixed 7-day template, no deadband",in-sample Round 1 reconstruction,87510.0,4.68,0.56,364,-5280.0,55390.0,1.58


,model,zero_shift_pnl,worst_shift_pnl,median_shift_pnl,max_drawdown_across_shifts,max_percentage_loss_vs_zero
0,Candidate A,88260.0,76680.0,83350.0,-8040.0,13.12
1,Candidate B,84040.0,76680.0,84040.0,-6510.0,8.76


,model,n_paths,median_pnl,p10_pnl,worst_pnl,positive_path_rate,median_max_drawdown
0,Candidate C,400,68025.95,48999.80,26739.05,1.0,-8034.40
1,Candidate D,400,66247.14,48119.00,29625.83,1.0,-7346.61
2,Constant AUD 45 mean reversion,400,63809.52,47495.50,29169.76,1.0,-11097.50
3,Candidate A,400,52305.65,30045.25,6591.43,1.0,-9473.45
4,"Fixed 7-day template, no deadband",400,51781.55,29313.69,-2410.24,1.0,-9961.79
5,Candidate B,400,50310.36,31474.58,14705.48,1.0,-8720.42
6,Broad fixed-day schedule,400,48517.62,33853.60,18537.14,1.0,-6126.67
7,Flat Boat Party position,400,0.00,0.00,0.00,0.0,0.00


,summer_equilibrium,model,n_paths,median_pnl,p10_pnl,worst_pnl,positive_path_rate,median_max_drawdown
0,45.0,Candidate C,100,69420.48,52646.71,37180.95,1.0,-7610.00
1,45.0,Candidate D,100,67140.24,52770.10,43175.24,1.0,-6910.71
2,43.0,Candidate C,100,56925.00,42266.57,17928.10,1.0,-7781.67
3,47.0,Candidate D,100,53299.05,37864.00,22826.67,1.0,-7275.00
4,43.0,Candidate A,100,53205.24,37413.90,20236.67,1.0,-8576.19
5,43.0,Candidate D,100,52330.48,37848.33,28269.05,1.0,-7732.86
6,45.0,Candidate A,100,50719.29,32715.57,22033.33,1.0,-8554.29
7,47.0,Candidate B,100,49987.38,34896.62,18545.71,1.0,-8509.52
8,43.0,Candidate B,100,49304.29,35093.38,25139.52,1.0,-8023.10
9,47.0,Candidate C,100,49282.38,37557.00,26374.76,1.0,-8371.90


,scenario_scope,target_equilibrium,transition_days,model,median_full_year_pnl,p10_full_year_pnl,worst_full_year_pnl,median_summer_only_pnl,p10_summer_only_pnl,worst_summer_only_pnl,median_max_drawdown,positive_path_rate,paired_median_d_minus_b,paired_p10_d_minus_b,paired_worst_d_minus_b
0,individual,43.0,7.0,Candidate B,49501.67,36121.29,22095.71,1589.52,-4953.62,-14047.14,-8022.38,1.0,1635.48,-6928.05,-19114.76
1,individual,43.0,7.0,Candidate D,51625.95,39021.76,27211.90,2796.43,-1267.43,-3407.14,-7536.43,1.0,1635.48,-6928.05,-19114.76
2,individual,43.0,14.0,Candidate B,48215.95,34835.57,20810.00,303.81,-6239.33,-15332.86,-8323.10,1.0,4677.38,-4782.52,-14841.90
3,individual,43.0,14.0,Candidate D,53671.90,39866.76,27069.05,4454.05,-890.71,-2450.00,-7421.67,1.0,4677.38,-4782.52,-14841.90
4,individual,43.0,28.0,Candidate B,48287.38,34907.00,20881.43,375.24,-6167.90,-15261.43,-8312.38,1.0,6558.57,-3010.86,-14984.76
5,individual,43.0,28.0,Candidate D,55200.24,42484.48,28523.33,6886.19,1118.86,-2982.38,-7216.67,1.0,6558.57,-3010.86,-14984.76
6,individual,45.0,7.0,Candidate B,48358.81,34978.43,20952.86,446.67,-6096.48,-15190.00,-8222.38,1.0,15806.90,8180.33,-828.10
7,individual,45.0,7.0,Candidate D,65566.67,51676.76,41870.95,16813.33,10566.10,4224.29,-7200.95,1.0,15806.90,8180.33,-828.10
8,individual,45.0,14.0,Candidate B,48358.81,34978.43,20952.86,446.67,-6096.48,-15190.00,-8222.38,1.0,15806.90,8180.33,-828.10
9,individual,45.0,14.0,Candidate D,65566.67,51676.76,41870.95,16813.33,10566.10,4224.29,-7200.95,1.0,15806.90,8180.33,-828.10


,scenario_scope,target_equilibrium,transition_days,n_paths,paired_median_d_minus_b,paired_p10_d_minus_b,paired_worst_d_minus_b
0,individual,43.0,7.0,200,1635.48,-6928.05,-19114.76
1,individual,43.0,14.0,200,4677.38,-4782.52,-14841.90
2,individual,43.0,28.0,200,6558.57,-3010.86,-14984.76
3,individual,45.0,7.0,200,15806.90,8180.33,-828.10
4,individual,45.0,14.0,200,15806.90,8180.33,-828.10
5,individual,45.0,28.0,200,15806.90,8180.33,-828.10
6,individual,47.0,7.0,200,2142.62,-4739.81,-10271.90
7,individual,47.0,14.0,200,2667.62,-3797.90,-10897.62
8,individual,47.0,28.0,200,5532.38,-2401.48,-9451.43
9,pooled,NaN,NaN,1800,8169.52,-3341.38,-19114.76


,selection_check,passed,evidence
0,pooled P10 D >= B,1,D=42522.76; B=35002.86
1,pooled worst D >= B,1,D=24294.76; B=19810.00
2,every paired scenario median D-B >= -2000,1,worst scenario median=1635.48
3,observed Round 1 D summer overlay positive,1,summer-only P&L=12550.00
4,all correctness and budget checks pass,1,passed=12/12
5,selected candidate,1,Candidate D


Final fixed 5/7/11 audit complete; outputs written under research/boat_party/results


## Final adaptive EWMA overlay audit (research-only)

This section audits the reported adaptive EWMA overlay against frozen Candidate D. The repository contains the frozen `BOAT_PARTY_SEMESTER_SIGNALS` string, not a literal `BOAT_PARTY_SIGNALS` teammate implementation; the audit records that provenance. All Round 1 results are in-sample diagnostics, and synthetic/bootstrap results are generator-conditioned stress tests rather than confidence intervals.

In [12]:
from ewma_overlay_audit import run_audit

ewma_audit = run_audit(root, n_bootstrap_paths=200, n_placebo_paths=250, n_summer_paths=100, seed=20260810)
display(ewma_audit['comparison'][['strategy', 'pnl', 'max_drawdown', 'active_days', 'trade_count', 'turnover_units', 'max_capital']].round(2))
display(ewma_audit['regressions'][['strategy', 'sample', 'n', 'beta_deviation', 'standard_error', 'ci_lower_95', 'ci_upper_95', 'rank_correlation', 'directional_hit_rate']].round(4))
display(ewma_audit['checks'][['check', 'passed', 'evidence']])
print('EWMA overlay audit complete; outputs written under research/boat_party/results')

,strategy,pnl,max_drawdown,active_days,trade_count,turnover_units,max_capital
0,Frozen Candidate D,92560.0,-2460.0,333,92,134000,55390.0
1,Noisy signal flat neutral,80010.0,-2460.0,291,65,82000,55390.0
2,Adaptive EWMA alpha 0.65,101830.0,-3080.0,354,74,132000,55390.0
3,Adaptive EWMA alpha 0.90,97050.0,-2460.0,343,77,122000,55390.0
4,Seasonal template only,80010.0,-2460.0,291,65,82000,55390.0
5,EWMA overlay only,83150.0,-8860.0,326,195,334000,55390.0
6,Candidate D no summer,80010.0,-2460.0,291,65,82000,55390.0
7,Summer AUD 45 only,12550.0,-940.0,42,27,52000,46100.0
8,Signal plus summer AUD 45,92560.0,-2460.0,333,92,134000,55390.0
9,Reverse latest return on neutral,97320.0,-3080.0,363,72,142000,55390.0


,strategy,sample,n,beta_deviation,standard_error,ci_lower_95,ci_upper_95,rank_correlation,directional_hit_rate
0,Adaptive EWMA alpha 0.65,all_neutral_semester_days,31,-2.6141,0.5580,-3.7077,-1.5205,-0.6331,0.6452
1,Adaptive EWMA alpha 0.65,exclude_top_1_neutral_strategy_days,30,-2.3872,0.5317,-3.4293,-1.3451,-0.6089,0.6333
2,Adaptive EWMA alpha 0.65,exclude_top_5_neutral_strategy_days,26,-1.8461,0.4654,-2.7582,-0.9340,-0.5159,0.5769
3,Adaptive EWMA alpha 0.65,exclude_top_10_neutral_strategy_days,21,-1.0832,0.4545,-1.9739,-0.1925,-0.3117,0.4762
4,Adaptive EWMA alpha 0.65,semester_1,18,-2.5741,0.7353,-4.0153,-1.1329,-0.6140,0.6667
5,Adaptive EWMA alpha 0.65,semester_2,11,-2.0185,0.7672,-3.5222,-0.5147,-0.4273,0.5455
6,Adaptive EWMA alpha 0.65,block_0,6,-3.5674,0.8169,-5.1686,-1.9662,-0.9429,0.6667
7,Adaptive EWMA alpha 0.65,block_1,9,-2.8342,1.0107,-4.8151,-0.8532,-0.7500,0.7778
8,Adaptive EWMA alpha 0.65,block_2,3,NaN,NaN,NaN,NaN,0.5000,0.3333
9,Adaptive EWMA alpha 0.65,block_3,6,-1.8650,0.8353,-3.5022,-0.2278,-0.3714,0.6667


,check,passed,evidence
0,signal_string_length_is_322,1,length=322
1,signal_string_only_contains_plus_minus_zero,1,"characters=['+', '-', '0']"
2,available_signal_matches_candidate_B_before_su...,1,available BOAT_PARTY_SEMESTER_SIGNALS vs froze...
3,adaptive_positions_are_integer,1,pandas position dtype
4,all_positions_within_plus_minus_1000,1,adaptive alpha 0.65/0.90 paths
5,candidate_D_positions_within_limit_and_budget,1,max capital=55390.00
6,adaptive_max_capital_below_portfolio_cap,1,standalone Boat Party notional
7,toy_t_to_t_plus_1_alignment,1,"daily=[1000.0, 1000.0, 2000.0]; total=4000"
8,backtest_daily_pnl_equals_position_t_times_nex...,1,all alpha 0.65 days
9,no_extra_final_day_return,1,day 364 is explicitly flat and has no t+1 return


EWMA overlay audit complete; outputs written under research/boat_party/results
